# Data Cleaning & Quality Audit Pipeline

This notebook cleans, validates, recovers, imputes, and enriches all five datasets from `Data Before cleaning/` and writes clean outputs plus audit artifacts into `Data After cleaning/`.

### Problems Solved vs. Original Script
1. **Portable Relative Paths**: Automatically resolves `Data Before cleaning/` and `Data After cleaning/` whether executed from the repo root or `Data After cleaning/`.
2. **Master & Relational Integrity**: Validates both `Product_ID` (`PX01`, `PX02`, `P555`, `P666`) AND `Customer_ID` (`C5555`, `C6666`) against master tables.
3. **Unambiguous Sibling `Order_ID` Recovery**: Recovers missing/corrupted `Customer_ID`, `Order_Date` (e.g. `ORD10052`), and `Ship_Date` on multi-line orders from valid sibling rows when unambiguous, tracking changes in `Recovered_Fields`.
4. **Status vs. `Ship_Date` Reconciliation**: Rejects chronological impossibilities (`Ship_Date < Order_Date`), clears bogus `Ship_Date` on `Pending`/`Cancelled` orders (`Ship_Date_Cleared`), flags remaining `Shipped`/`Delivered` lines without `Ship_Date` (`Ship_Date_Anomaly`), and flags mixed-status orders (`Order_Status_Conflict`).
5. **Transparent Numeric Imputation**: Imputes missing `Unit_Price` via product median valid price (fallback to `Unit_Cost` markup) and missing `Shipping_Cost` via unambiguous sibling value or dataset median, tracked via `Is_Imputed` and `Imputed_Fields`.
6. **Quarantine Audit & KPI Enrichment**: Exports every rejected row to `rejected_records.csv` with exact `Rejection_Reason`, enriches valid rows with `Gross_Revenue`, `Net_Revenue`, `Line_Cost`, `Line_Profit`, `Delivery_Days`, and `Stock_Alert`, and generates `cleaning_summary.csv`.

In [1]:
import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd

ALLOWED_REGIONS = {"Cairo", "Giza", "Alexandria", "Mansoura"}
ALLOWED_SEGMENTS = {"Regular", "Silver", "Gold"}
ALLOWED_CATEGORIES = {"Electronics", "Home", "Beauty", "Fashion", "Sports"}
ALLOWED_STATUSES = {"Pending", "Shipped", "Delivered", "Cancelled"}
ALLOWED_PAYMENTS = {"Cash", "Card", "Wallet", "Online"}
UNSHIPPED_STATUSES = {"Pending", "Cancelled"}
SHIPPED_STATUSES = {"Shipped", "Delivered"}


def resolve_project_root() -> Path:
    """Locate the workspace root containing 'Data Before cleaning'."""
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "Data Before cleaning").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate 'Data Before cleaning' folder from current working directory.")


PROJECT_ROOT = resolve_project_root()
RAW_DIR = PROJECT_ROOT / "Data Before cleaning"
CLEAN_DIR = PROJECT_ROOT / "Data After cleaning"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("data_cleaner")
logger.info("RAW_DIR=%s | CLEAN_DIR=%s", RAW_DIR, CLEAN_DIR)

19:43:32 | INFO | RAW_DIR=C:\Users\L0Q\Desktop\scraping\Data Before cleaning | CLEAN_DIR=C:\Users\L0Q\Desktop\scraping\Data After cleaning


In [2]:
def clean_customers(raw_dir: Path, clean_dir: Path) -> tuple[pd.DataFrame, dict]:
    """Clean and validate customers_master.csv."""
    df = pd.read_csv(raw_dir / "customers_master.csv")
    raw_rows = len(df)

    df["Customer_ID"] = df["Customer_ID"].astype(str).str.strip().str.upper()
    df["Customer_Name"] = df["Customer_Name"].astype(str).str.strip()
    df["Region"] = df["Region"].astype(str).str.strip().str.title()
    df["Customer_Segment"] = df["Customer_Segment"].astype(str).str.strip().str.title()

    df = df.drop_duplicates().reset_index(drop=True)
    df = df.drop_duplicates(subset=["Customer_ID"], keep="first").reset_index(drop=True)

    assert df["Customer_ID"].str.match(r"^C\d{4}$").all(), "Unexpected Customer_ID format"
    assert set(df["Region"]).issubset(ALLOWED_REGIONS), f"Unexpected regions: {set(df['Region']) - ALLOWED_REGIONS}"
    assert set(df["Customer_Segment"]).issubset(ALLOWED_SEGMENTS), "Unexpected segments"

    df.to_csv(clean_dir / "customers_master_cleaned.csv", index=False)
    stats = {
        "dataset": "customers_master",
        "raw_rows": raw_rows,
        "duplicates_removed": raw_rows - len(df),
        "recovered_rows": 0,
        "imputed_rows": 0,
        "ship_date_cleared_rows": 0,
        "ship_date_anomaly_rows": 0,
        "rejected_rows": 0,
        "clean_rows": len(df),
        "net_revenue": np.nan,
        "line_profit": np.nan,
    }
    return df, stats


def clean_products(raw_dir: Path, clean_dir: Path) -> tuple[pd.DataFrame, dict]:
    """Clean, validate, and enrich products_master.csv with stock alert indicators."""
    df = pd.read_csv(raw_dir / "products_master.csv")
    raw_rows = len(df)

    df["Product_ID"] = df["Product_ID"].astype(str).str.strip().str.upper()
    df["Product_Name"] = df["Product_Name"].astype(str).str.strip()
    df["Category"] = df["Category"].astype(str).str.strip().str.title()
    for col in ("Unit_Cost", "Current_Stock", "Reorder_Level"):
        df[col] = pd.to_numeric(df[col], errors="raise")

    df = df.drop_duplicates().reset_index(drop=True)
    df = df.drop_duplicates(subset=["Product_ID"], keep="first").reset_index(drop=True)

    assert df["Product_ID"].str.match(r"^P\d{3}$").all(), "Unexpected Product_ID format"
    assert set(df["Category"]).issubset(ALLOWED_CATEGORIES), f"Unexpected categories: {set(df['Category']) - ALLOWED_CATEGORIES}"
    assert (df[["Unit_Cost", "Current_Stock", "Reorder_Level"]] >= 0).all().all()

    df["Stock_Alert"] = np.select(
        [df["Current_Stock"] == 0, df["Current_Stock"] < df["Reorder_Level"]],
        ["Out of Stock", "Reorder Needed"],
        default="Healthy",
    )
    df["Units_Below_Reorder"] = (df["Reorder_Level"] - df["Current_Stock"]).clip(lower=0).astype(int)

    df.to_csv(clean_dir / "products_master_cleaned.csv", index=False)
    stats = {
        "dataset": "products_master",
        "raw_rows": raw_rows,
        "duplicates_removed": raw_rows - len(df),
        "recovered_rows": 0,
        "imputed_rows": 0,
        "ship_date_cleared_rows": 0,
        "ship_date_anomaly_rows": 0,
        "rejected_rows": 0,
        "clean_rows": len(df),
        "net_revenue": np.nan,
        "line_profit": np.nan,
    }
    return df, stats


def clean_previous_summary(raw_dir: Path, clean_dir: Path) -> tuple[pd.DataFrame, dict]:
    """Clean and validate previous_days_summary.csv."""
    df = pd.read_csv(raw_dir / "previous_days_summary.csv")
    raw_rows = len(df)

    df["Date"] = pd.to_datetime(df["Date"], errors="raise")
    df = df.drop_duplicates().sort_values("Date").reset_index(drop=True)

    assert df["Date"].is_unique, "Duplicate dates in previous_days_summary"
    assert (df["Record_Count"] >= df["Total_Orders"]).all()
    assert (df["Rejected_Records"] >= 0).all()
    for rate_col in ("Valid_Records_Rate", "Missing_Rate", "Duplicate_Rate"):
        assert ((df[rate_col] >= 0) & (df[rate_col] <= 1)).all(), f"Rate out of [0,1]: {rate_col}"

    df.to_csv(clean_dir / "previous_days_summary_cleaned.csv", index=False)
    stats = {
        "dataset": "previous_days_summary",
        "raw_rows": raw_rows,
        "duplicates_removed": raw_rows - len(df),
        "recovered_rows": 0,
        "imputed_rows": 0,
        "ship_date_cleared_rows": 0,
        "ship_date_anomaly_rows": 0,
        "rejected_rows": 0,
        "clean_rows": len(df),
        "net_revenue": round(float(df["Total_Sales"].sum()), 2),
        "line_profit": np.nan,
    }
    return df, stats

In [3]:
def normalize_raw_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize strings, numeric formats ('pcs', '%'), and dates without dropping rows."""
    out = df.copy()
    out["Original_Row_Number"] = np.arange(2, len(out) + 2)

    for col in ("Order_ID", "Customer_ID", "Product_ID"):
        out[col] = out[col].where(out[col].notna(), np.nan)
        mask = out[col].notna()
        out.loc[mask, col] = out.loc[mask, col].astype(str).str.strip().str.upper()
        out.loc[out[col] == "", col] = np.nan

    for col in ("Order_Status", "Payment_Method"):
        out[col] = out[col].astype(str).str.strip().str.title()

    qty_clean = (
        out["Quantity"]
        .astype(str)
        .str.replace(r"\s*pcs$", "", case=False, regex=True)
        .str.strip()
    )
    out["Quantity"] = pd.to_numeric(qty_clean, errors="coerce")
    out["Unit_Price"] = pd.to_numeric(out["Unit_Price"], errors="coerce")
    out["Shipping_Cost"] = pd.to_numeric(out["Shipping_Cost"], errors="coerce")

    disc_str = out["Discount"].astype(str).str.strip()
    is_pct = disc_str.str.contains("%", regex=False, na=False)
    out["Discount"] = pd.to_numeric(disc_str.str.replace("%", "", regex=False), errors="coerce")
    out.loc[is_pct, "Discount"] = out.loc[is_pct, "Discount"] / 100.0

    out["Order_Date"] = pd.to_datetime(out["Order_Date"], errors="coerce")
    out["Ship_Date"] = pd.to_datetime(out["Ship_Date"], errors="coerce")
    return out


def build_unambiguous_sibling_map(df: pd.DataFrame, column: str, valid_mask: pd.Series) -> dict:
    """Map Order_ID -> single unique valid value from sibling rows when unambiguous."""
    valid_subset = df.loc[valid_mask, ["Order_ID", column]].dropna()
    grouped = valid_subset.groupby("Order_ID")[column].nunique()
    unambiguous_ids = grouped[grouped == 1].index
    return (
        valid_subset[valid_subset["Order_ID"].isin(unambiguous_ids)]
        .drop_duplicates(subset=["Order_ID"])
        .set_index("Order_ID")[column]
        .to_dict()
    )


def recover_from_siblings(df: pd.DataFrame, valid_customers: set[str]) -> tuple[pd.DataFrame, pd.Series]:
    """Recover missing/corrupted order-level attributes from unambiguous sibling Order_ID rows."""
    out = df.copy()
    recovered_tags = pd.Series([[] for _ in range(len(out))], index=out.index, dtype=object)

    # 1) Customer_ID recovery (missing or invalid FK such as C5555/C6666 when siblings are valid & unique)
    cust_valid = out["Customer_ID"].isin(valid_customers)
    cust_map = build_unambiguous_sibling_map(out, "Customer_ID", cust_valid)
    need_cust = (~cust_valid) & out["Order_ID"].isin(cust_map)
    if need_cust.any():
        out.loc[need_cust, "Customer_ID"] = out.loc[need_cust, "Order_ID"].map(cust_map)
        for idx in out.index[need_cust]:
            recovered_tags[idx].append("Customer_ID")

    # 2) Order_Date recovery (e.g., ORD10052 'not-a-date' recovered from sibling row)
    date_valid = out["Order_Date"].notna()
    date_map = build_unambiguous_sibling_map(out, "Order_Date", date_valid)
    need_date = (~date_valid) & out["Order_ID"].isin(date_map)
    if need_date.any():
        out.loc[need_date, "Order_Date"] = out.loc[need_date, "Order_ID"].map(date_map)
        for idx in out.index[need_date]:
            recovered_tags[idx].append("Order_Date")

    # 3) Ship_Date recovery for Shipped/Delivered rows when sibling rows share one valid Ship_Date >= Order_Date
    ship_valid = (
        out["Order_Status"].isin(SHIPPED_STATUSES)
        & out["Ship_Date"].notna()
        & out["Order_Date"].notna()
        & (out["Ship_Date"] >= out["Order_Date"])
    )
    ship_map = build_unambiguous_sibling_map(out, "Ship_Date", ship_valid)
    need_ship = (
        out["Order_Status"].isin(SHIPPED_STATUSES)
        & out["Ship_Date"].isna()
        & out["Order_ID"].isin(ship_map)
    )
    if need_ship.any():
        candidate_dates = out.loc[need_ship, "Order_ID"].map(ship_map)
        safe = candidate_dates >= out.loc[need_ship, "Order_Date"]
        safe_idx = out.index[need_ship][safe]
        out.loc[safe_idx, "Ship_Date"] = candidate_dates.loc[safe_idx]
        for idx in safe_idx:
            recovered_tags[idx].append("Ship_Date")

    recovered_str = recovered_tags.apply(lambda tags: ", ".join(tags))
    return out, recovered_str


def build_rejection_reasons(
    df: pd.DataFrame,
    valid_customers: set[str],
    valid_products: set[str],
) -> pd.Series:
    """Evaluate hard validation rules per row and return semicolon-delimited rejection reasons."""
    checks = [
        (df["Order_ID"].isna(), "Missing Order_ID"),
        (~df["Customer_ID"].isin(valid_customers), "Invalid or missing Customer_ID"),
        (~df["Product_ID"].isin(valid_products), "Invalid Product_ID"),
        (df["Quantity"].isna() | (df["Quantity"] <= 0), "Quantity <= 0 or non-numeric"),
        (df["Discount"].isna() | (df["Discount"] < 0) | (df["Discount"] > 1), "Discount outside [0, 1]"),
        (df["Order_Date"].isna(), "Invalid or missing Order_Date"),
        (df["Order_Date"].notna() & df["Ship_Date"].notna() & (df["Ship_Date"] < df["Order_Date"]), "Ship_Date before Order_Date"),
        (~df["Order_Status"].isin(ALLOWED_STATUSES), "Invalid Order_Status"),
        (~df["Payment_Method"].isin(ALLOWED_PAYMENTS), "Invalid Payment_Method"),
    ]
    reasons = pd.Series([[] for _ in range(len(df))], index=df.index, dtype=object)
    for mask, message in checks:
        for idx in df.index[mask]:
            reasons[idx].append(message)
    return reasons.apply(lambda r: "; ".join(r))

In [4]:
def clean_orders_dataset(
    file_path: Path,
    source_name: str,
    valid_customers: set[str],
    valid_products: set[str],
    products_df: pd.DataFrame,
    global_price_map: dict[str, float],
    fallback_markup: float,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Full cleaning, sibling recovery, quarantine, imputation, and KPI enrichment for orders/batch."""
    raw_df = pd.read_csv(file_path)
    raw_rows = len(raw_df)
    norm_df = normalize_raw_orders(raw_df)

    # 1) Exact duplicates after normalizing whitespace/casing
    feature_cols = [c for c in norm_df.columns if c != "Original_Row_Number"]
    dup_mask = norm_df.duplicated(subset=feature_cols, keep="first")
    dup_rejected = norm_df.loc[dup_mask].copy()
    dup_rejected["Source_File"] = source_name
    dup_rejected["Rejection_Reason"] = "Exact duplicate row"

    work_df = norm_df.loc[~dup_mask].copy()

    # 2) Unambiguous sibling recovery
    work_df, recovered_series = recover_from_siblings(work_df, valid_customers)
    work_df["Recovered_Fields"] = recovered_series

    # 3) Quarantine hard rule failures
    rejection_reasons = build_rejection_reasons(work_df, valid_customers, valid_products)
    invalid_mask = rejection_reasons != ""
    rule_rejected = work_df.loc[invalid_mask].copy()
    rule_rejected["Source_File"] = source_name
    rule_rejected["Rejection_Reason"] = rejection_reasons.loc[invalid_mask]

    valid_df = work_df.loc[~invalid_mask].copy()

    # 4) Key-level duplicate check on (Order_ID, Product_ID)
    key_dup_mask = valid_df.duplicated(subset=["Order_ID", "Product_ID"], keep="first")
    key_rejected = valid_df.loc[key_dup_mask].copy()
    key_rejected["Source_File"] = source_name
    key_rejected["Rejection_Reason"] = "Duplicate (Order_ID, Product_ID) key"

    valid_df = valid_df.loc[~key_dup_mask].copy()

    # 5) Reconcile Order_Status vs. Ship_Date
    clear_mask = valid_df["Order_Status"].isin(UNSHIPPED_STATUSES) & valid_df["Ship_Date"].notna()
    valid_df["Ship_Date_Cleared"] = clear_mask
    valid_df.loc[clear_mask, "Ship_Date"] = pd.NaT

    valid_df["Ship_Date_Anomaly"] = valid_df["Order_Status"].isin(SHIPPED_STATUSES) & valid_df["Ship_Date"].isna()

    order_has_cancelled = valid_df.groupby("Order_ID")["Order_Status"].transform(lambda s: (s == "Cancelled").any())
    order_has_active = valid_df.groupby("Order_ID")["Order_Status"].transform(lambda s: (s != "Cancelled").any())
    valid_df["Order_Status_Conflict"] = order_has_cancelled & order_has_active

    # 6) Impute missing Unit_Price & Shipping_Cost with transparency flags
    imputed_tags = pd.Series([[] for _ in range(len(valid_df))], index=valid_df.index, dtype=object)

    cost_map = products_df.set_index("Product_ID")["Unit_Cost"].to_dict()
    local_price_map = valid_df.dropna(subset=["Unit_Price"]).groupby("Product_ID")["Unit_Price"].median().to_dict()
    merged_price_map = {**global_price_map, **local_price_map}

    miss_price = valid_df["Unit_Price"].isna()
    if miss_price.any():
        for idx in valid_df.index[miss_price]:
            pid = valid_df.at[idx, "Product_ID"]
            imputed_val = merged_price_map.get(pid, round(cost_map[pid] * fallback_markup, 2))
            valid_df.at[idx, "Unit_Price"] = round(float(imputed_val), 2)
            imputed_tags[idx].append("Unit_Price")

    sibling_ship_cost_map = build_unambiguous_sibling_map(valid_df, "Shipping_Cost", valid_df["Shipping_Cost"].notna())
    dataset_ship_median = round(float(valid_df["Shipping_Cost"].median()), 2)
    miss_ship = valid_df["Shipping_Cost"].isna()
    if miss_ship.any():
        for idx in valid_df.index[miss_ship]:
            oid = valid_df.at[idx, "Order_ID"]
            valid_df.at[idx, "Shipping_Cost"] = round(float(sibling_ship_cost_map.get(oid, dataset_ship_median)), 2)
            imputed_tags[idx].append("Shipping_Cost")

    valid_df["Imputed_Fields"] = imputed_tags.apply(lambda tags: ", ".join(tags))
    valid_df["Is_Imputed"] = valid_df["Imputed_Fields"] != ""

    # 7) Derived KPIs & clean dtypes
    valid_df["Quantity"] = valid_df["Quantity"].astype("Int64")
    valid_df["Unit_Price"] = valid_df["Unit_Price"].round(2)
    valid_df["Discount"] = valid_df["Discount"].round(4)
    valid_df["Shipping_Cost"] = valid_df["Shipping_Cost"].round(2)

    unit_cost_series = valid_df["Product_ID"].map(cost_map)
    valid_df["Gross_Revenue"] = (valid_df["Quantity"] * valid_df["Unit_Price"]).round(2)
    valid_df["Net_Revenue"] = (valid_df["Gross_Revenue"] * (1.0 - valid_df["Discount"])).round(2)
    valid_df["Line_Cost"] = (valid_df["Quantity"] * unit_cost_series).round(2)
    valid_df["Line_Profit"] = (valid_df["Net_Revenue"] - valid_df["Line_Cost"]).round(2)
    valid_df["Delivery_Days"] = (valid_df["Ship_Date"] - valid_df["Order_Date"]).dt.days.astype("Int64")

    valid_df = valid_df.drop(columns=["Original_Row_Number"]).reset_index(drop=True)

    rejected_all = pd.concat([dup_rejected, rule_rejected, key_rejected], ignore_index=True)
    if not rejected_all.empty:
        rejected_all = rejected_all.sort_values("Original_Row_Number").reset_index(drop=True)

    stats = {
        "dataset": source_name,
        "raw_rows": raw_rows,
        "duplicates_removed": int(len(dup_rejected) + len(key_rejected)),
        "recovered_rows": int((valid_df["Recovered_Fields"] != "").sum()),
        "imputed_rows": int(valid_df["Is_Imputed"].sum()),
        "ship_date_cleared_rows": int(valid_df["Ship_Date_Cleared"].sum()),
        "ship_date_anomaly_rows": int(valid_df["Ship_Date_Anomaly"].sum()),
        "rejected_rows": int(len(rejected_all)),
        "clean_rows": int(len(valid_df)),
        "net_revenue": round(float(valid_df["Net_Revenue"].sum()), 2),
        "line_profit": round(float(valid_df["Line_Profit"].sum()), 2),
    }
    return valid_df, rejected_all, stats

In [5]:
customers_df, cust_stats = clean_customers(RAW_DIR, CLEAN_DIR)
products_df, prod_stats = clean_products(RAW_DIR, CLEAN_DIR)
summary_df, prev_stats = clean_previous_summary(RAW_DIR, CLEAN_DIR)

valid_customers = set(customers_df["Customer_ID"])
valid_products = set(products_df["Product_ID"])

# Build global product median price & markup reference across raw valid lines for Unit_Price fallback
orders_norm_ref = normalize_raw_orders(pd.read_csv(RAW_DIR / "orders.csv"))
batch_norm_ref = normalize_raw_orders(pd.read_csv(RAW_DIR / "new_daily_batch.csv"))
combined_ref = pd.concat([orders_norm_ref, batch_norm_ref], ignore_index=True)
combined_ref = combined_ref[combined_ref["Product_ID"].isin(valid_products) & combined_ref["Unit_Price"].notna()]
global_price_map = combined_ref.groupby("Product_ID")["Unit_Price"].median().to_dict()
cost_lookup = products_df.set_index("Product_ID")["Unit_Cost"].to_dict()
fallback_markup = float((combined_ref["Unit_Price"] / combined_ref["Product_ID"].map(cost_lookup)).median())

orders_clean, orders_rejected, orders_stats = clean_orders_dataset(
    RAW_DIR / "orders.csv",
    "orders.csv",
    valid_customers,
    valid_products,
    products_df,
    global_price_map,
    fallback_markup,
)
orders_clean.to_csv(CLEAN_DIR / "orders_cleaned.csv", index=False)

batch_clean, batch_rejected, batch_stats = clean_orders_dataset(
    RAW_DIR / "new_daily_batch.csv",
    "new_daily_batch.csv",
    valid_customers,
    valid_products,
    products_df,
    global_price_map,
    fallback_markup,
)
batch_clean.to_csv(CLEAN_DIR / "new_daily_batch_cleaned.csv", index=False)

# Combine & export rejected quarantine audit
rejected_all = pd.concat([orders_rejected, batch_rejected], ignore_index=True)
audit_cols = [
    "Source_File", "Original_Row_Number", "Rejection_Reason", "Order_ID",
    "Order_Date", "Ship_Date", "Customer_ID", "Product_ID", "Quantity",
    "Unit_Price", "Discount", "Shipping_Cost", "Order_Status", "Payment_Method",
]
rejected_all = rejected_all[[c for c in audit_cols if c in rejected_all.columns]]
rejected_all.to_csv(CLEAN_DIR / "rejected_records.csv", index=False)

# Export summary report
summary_report_df = pd.DataFrame([cust_stats, prod_stats, prev_stats, orders_stats, batch_stats])
summary_report_df.to_csv(CLEAN_DIR / "cleaning_summary.csv", index=False)

print("=== Cleaning Summary ===")
print(summary_report_df.to_string(index=False))
print("\n=== Rejection Breakdown ===")
print(rejected_all.groupby(["Source_File", "Rejection_Reason"]).size().to_string())

=== Cleaning Summary ===
              dataset  raw_rows  duplicates_removed  recovered_rows  imputed_rows  ship_date_cleared_rows  ship_date_anomaly_rows  rejected_rows  clean_rows  net_revenue  line_profit
     customers_master       150                   0               0             0                       0                       0              0         150          NaN          NaN
      products_master        30                   0               0             0                       0                       0              0          30          NaN          NaN
previous_days_summary        30                   0               0             0                       0                       0              0          30   8418231.21          NaN
           orders.csv      1020                   2               4             2                       0                       1              9        1011   1055953.26    273418.26
  new_daily_batch.csv       333                   4         